# **Add county information to `solar_lcoe_ReEDS.csv`**

This notebook adds county information to the solar metadata file `solar_lcoe_ReEDS.csv` using a polygon-to-polygon matching workflow, then filters the final outputs to the requested WECC states.

The process is:
1. load `solar_lcoe_ReEDS.csv`
2. load the solar CPA polygons from the Zenodo candidate project area shapefiles
3. load the TIGER/Line county shapefile
4. spatially match each CPA polygon to the counties it intersects
5. for CPAs that cross multiple counties, assign a primary county using the largest overlap area
6. merge the county assignment back into `solar_lcoe_ReEDS.csv`
7. filter the final outputs to the requested WECC states
8. save the updated metadata file and QA outputs

The final result is a WECC-filtered, county-enriched solar metadata file that can be used in the next clustering step.

## **Folder structure expected by this notebook**

Place the input files here inside the repo:

```text
solar-county-analysis/
  data/
    reeds_county_mapping/
      inputs/
        solar_lcoe_ReEDS.csv
        CandidateProjectAreas_WindAndSolar_20210623/
          CandidateProjectArea_SolarPV.shp
          CandidateProjectArea_SolarPV.shx
          CandidateProjectArea_SolarPV.dbf
          CandidateProjectArea_SolarPV.prj
          ...
        tl_2024_us_county/
          tl_2024_us_county.shp
          tl_2024_us_county.shx
          tl_2024_us_county.dbf
          tl_2024_us_county.prj
          ...
      outputs/
  notebooks/
    reeds_add_counties_to_solar_lcoe.ipynb

In [22]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 160)

In [23]:
NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = REPO_ROOT / "data" / "reeds_county_mapping"
INPUT_DIR = DATA_DIR / "inputs"
OUTPUT_DIR = DATA_DIR / "outputs"

SOLAR_METADATA_CSV = INPUT_DIR / "solar_lcoe_ReEDS.csv"

CPA_DIR = INPUT_DIR / "CandidateProjectAreas_WindAndSolar_20210623"
CPA_SHP = CPA_DIR / "CandidateProjectArea_SolarPV.shp"

COUNTY_DIR = INPUT_DIR / "tl_2024_us_county"
COUNTY_SHP = COUNTY_DIR / "tl_2024_us_county.shp"

COUNTY_GEOJSON_OPTIONAL = REPO_ROOT / "county_layer_for_gui.geojson"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

paths_to_check = {
    "repo_root": REPO_ROOT,
    "data_dir": DATA_DIR,
    "solar_metadata_csv": SOLAR_METADATA_CSV,
    "cpa_shapefile": CPA_SHP,
    "county_shapefile": COUNTY_SHP,
    "county_geojson_optional": COUNTY_GEOJSON_OPTIONAL,
    "output_dir": OUTPUT_DIR,
}

for name, p in paths_to_check.items():
    print(f"{name}: {p.exists()} -> {p}")

repo_root: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores
data_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping
solar_metadata_csv: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/solar_lcoe_ReEDS.csv
cpa_shapefile: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/CandidateProjectAreas_WindAndSolar_20210623/CandidateProjectArea_SolarPV.shp
county_shapefile: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/tl_2024_us_county/tl_2024_us_county.shp
county_geojson_optional: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/county_layer_for_gui.geojson
output_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/outputs


In [24]:
# Load the three core input datasets needed for this workflow.

# solar_meta:
# This is the main metadata table we want to modify.
# Source file:
#   solar_lcoe_ReEDS.csv
#
# It contains one row per solar Candidate Project Area (CPA) record used in the
# Switch-PG-ReEDS / PowerGenome workflow. The goal of this notebook is to add
# county information into this metadata table so it can later be used for
# county-based renewable clustering.
solar_meta = pd.read_csv(SOLAR_METADATA_CSV, low_memory=False)

# cpa:
# This loads the solar CPA polygons from the Zenodo shapefile bundle.
# Source file:
#   CandidateProjectArea_SolarPV.shp
#
# Each row here is a geographic polygon representing a solar Candidate Project Area.
# This shapefile is what allows us to connect each CPA_ID to an actual location on the map.
# Later in the notebook, these polygons will be spatially joined to county polygons.
cpa = gpd.read_file(CPA_SHP)

# counties:
# This loads the county boundary shapefile, ideally from TIGER/Line.
# Source file:
#   tl_2024_us_county.shp
#
# Each row here is a county polygon. This layer contains the county boundaries
# we will spatially join against the CPA polygons in order to determine which
# county each CPA belongs to.
counties = gpd.read_file(COUNTY_SHP)

# -----------------------------
# Quick QA / inspection prints
# -----------------------------

print("solar_meta shape:", solar_meta.shape)
print("solar_meta columns:", solar_meta.columns.tolist())
print("\ncpa shape:", cpa.shape)
print("cpa columns:", cpa.columns.tolist())
print("cpa CRS:", cpa.crs)
print("\ncounties shape:", counties.shape)
print("counties columns:", counties.columns.tolist())
print("counties CRS:", counties.crs)

display(solar_meta.head())
display(cpa.head())
display(counties.head())

solar_meta shape: (405737, 43)
solar_meta columns: ['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFarmland', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_min', 'Shape_Leng', 'm_landcover', 'exFacil', 'plFacil', 'Qual_Coal', 'Qual_Emp', 'Qual_Brown', 'anyQual', 'Qual_noBF', 'SocialImpa', 'EnviroImpa', 'Shape_Le_1', 'Shape_Area', 'pop_density_bin', 'tech', 'metro_id', 'metro_region', 'cpa_mw', 'cf', 'path', 'resource_annuity', 'resource_fom', 'interconnect_annuity', 'lcoe', 'interconnect_capex_mw', 'total_interconnect_km', 'offshore_interconnect_km', 'ipm_region']

cpa shape: (406110, 22)
cpa columns: ['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFar', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_m', 'Shape_Leng', 'Shape_Area', 'm_landcove', 'exFacil', 'plFacil', 'cf', 'geometry']
cpa CRS: PROJCS["NAD_1983_Albers",GEOGCS["NAD83",DATUM["North_America

,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFar,incap,CPA_ID,m_aspect,m_aspect_m,Shape_Leng,Shape_Area,m_landcove,exFacil,plFacil,cf,geometry
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,2.000000e+06,81,0,0,0.220263,"POLYGON ((-1924591.227 3153922.078, -1924591.227 3153422.078, -1924091.227 3153422.078, -1924091.227 3152422.078, -1925591.227 3152422.078, -1925591.227 315..."
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,3.750000e+06,81,0,0,0.224177,"POLYGON ((-1929591.227 3151922.078, -1929591.227 3151422.078, -1929091.227 3151422.078, -1929091.227 3150422.078, -1929591.227 3150422.078, -1929591.227 314..."
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,3.000000e+06,42,0,0,0.222078,"POLYGON ((-1900091.227 3148922.078, -1900091.227 3148422.078, -1899591.227 3148422.078, -1899591.227 3146922.078, -1900591.227 3146922.078, -1900591.227 314..."
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,2.000000e+06,81,0,0,0.223520,"POLYGON ((-1914091.227 3143922.078, -1914091.227 3142922.078, -1913591.227 3142922.078, -1913591.227 3142422.078, -1915091.227 3142422.078, -1915091.227 314..."
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,2.250000e+06,42,0,0,0.223520,"POLYGON ((-1915591.227 3139422.078, -1915591.227 3138922.078, -1916591.227 3138922.078, -1916591.227 3139422.078, -1917091.227 3139422.078, -1917091.227 313..."


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,0500000US31039,Cuming,Cuming County,06,H1,G4020,NaN,NaN,NaN,A,1477563042,10772508,+41.9158651,-096.7885168,"POLYGON ((-96.55525 41.82892, -96.55524 41.82758, -96.55524 41.82753, -96.55524 41.82739, -96.55524 41.8243, -96.55523 41.82217, -96.55524 41.82037, -96.555..."
1,53,069,01513275,53069,0500000US53069,Wahkiakum,Wahkiakum County,06,H1,G4020,NaN,NaN,NaN,A,680980773,61564428,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26476, -123.72768 46.27377, -123.72773 46.27788, -123.72774 46.27872, -123.72783 46.28508, -123.72788 46.28834, ..."
2,35,011,00933054,35011,0500000US35011,De Baca,De Baca County,06,H1,G4020,NaN,NaN,NaN,A,6016818941,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.08908, -104.89334 34.09434, -104.89334 34.09458, -104.89334 34.09481, -104.89331 34.10002, -104.89329 34.10286,..."
3,31,109,00835876,31109,0500000US31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,NaN,A,2169269508,22850511,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.52312, -96.69369 40.52309, -96.6944 40.52307, -96.69461 40.52307, -96.70282 40.52307, -96.7029 40.52307, -96.70418..."
4,31,129,00835886,31129,0500000US31129,Nuckolls,Nuckolls County,06,H1,G4020,NaN,NaN,NaN,A,1489645201,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224, -98.27374 40.12253, -98.27375 40.12314, -98.27378 40.12501, -98.2737 40.13304, -98.27369 40.13943, -98.27369 ..."


## Notes on the matching logic

`solar_lcoe_ReEDS.csv` already contains a unique `CPA_ID` for each solar candidate project area row. The Zenodo solar shapefile should also contain `CPA_ID`, which is the join key linking the metadata CSV to the CPA polygons.

The county shapefile contains county polygons with fields like:
- `GEOID` = county FIPS
- `NAME` = county name
- `NAMELSAD` = county name with legal/statistical area description
- `STATEFP` = state FIPS

The goal is to assign a **primary county** to each CPA so the updated metadata file can be used in county-based clustering. Because some CPA polygons may cross county boundaries, this notebook first finds all counties each CPA intersects, then computes the overlap area, and finally assigns the CPA to the county with the **largest overlap area**.

In [25]:
# -----------------------------
# Reduce both geospatial layers down to only the fields needed for county assignment
# -----------------------------

# The full CPA shapefile and full county shapefile contain many extra columns
# that are not needed for the actual matching workflow.
# To keep the geospatial work easier to follow and a little lighter to run,
# this step trims each GeoDataFrame down to the minimum fields we care about.

# For the CPA layer, we only keep:
# - CPA_ID: the unique identifier that links each CPA polygon back to solar_lcoe_ReEDS.csv
# - geometry: the actual polygon shape for each solar Candidate Project Area
#
# We do not need the other CPA attributes yet because this step is only about
# figuring out which county each CPA belongs to.
cpa = cpa[["CPA_ID", "geometry"]].copy()

# For the county layer, we keep only the fields needed to identify the county
# after the spatial join:
# - GEOID: county FIPS code
# - NAME: short county name
# - NAMELSAD: fuller county name with legal/statistical designation
# - STATEFP: state FIPS code
# - geometry: county polygon
#
# These are enough to create a county assignment for each CPA and to preserve
# stable geographic identifiers for QA and downstream use.
counties = counties[["GEOID", "NAME", "NAMELSAD", "STATEFP", "geometry"]].copy()

# Rename the county fields to clearer, downstream-friendly names.
# This makes later merge and output steps easier to read.
# After renaming:
# - county_fips is the county identifier
# - county_name is the short county name
# - county_name_full includes the legal/statistical area description
# - state_fips is the state FIPS code
counties = counties.rename(columns={
    "GEOID": "county_fips",
    "NAME": "county_name",
    "NAMELSAD": "county_name_full",
    "STATEFP": "state_fips",
})

# -----------------------------
# Reproject both layers into a projected CRS for area-based overlap calculations
# -----------------------------

# The CPA and county files may start in a geographic CRS like EPSG:4326,
# where coordinates are stored as longitude/latitude degrees.
# That is fine for map display, but not ideal for measuring polygon overlap area.
#
# Because this workflow assigns each CPA to the county with the *largest overlap area*,
# the area calculations need to happen in a projected coordinate system, not in degrees.

# EPSG:5070 is a commonly used U.S. Albers equal-area projection.
# It is a good choice here because:
# - it is designed for the United States
# - area calculations are much more meaningful
# - it keeps the CPA/county overlap comparison consistent across the country
#
# This does NOT change the meaning of the geometries.
# It only changes the coordinate system used internally for geometric operations.
cpa = cpa.to_crs("EPSG:5070")
counties = counties.to_crs("EPSG:5070")

print("CPA CRS after reprojection:", cpa.crs)
print("County CRS after reprojection:", counties.crs)
print("Unique CPA_IDs in metadata:", solar_meta["CPA_ID"].nunique())
print("Unique CPA_IDs in shapefile:", cpa["CPA_ID"].nunique())

CPA CRS after reprojection: EPSG:5070
County CRS after reprojection: EPSG:5070
Unique CPA_IDs in metadata: 405737
Unique CPA_IDs in shapefile: 406110


In [26]:
# -----------------------------
# Spatially match each CPA polygon to every county polygon it touches
# -----------------------------

# This is the first actual geospatial matching step in the workflow.
# At this point:
# - `cpa` contains the solar Candidate Project Area polygons from the Zenodo shapefile
# - `counties` contains the county boundary polygons from the TIGER/Line shapefile
#
# The goal here is NOT yet to decide the final county for each CPA.
# Instead, the goal is to find all counties that each CPA could possibly belong to.

# gpd.sjoin(..., predicate="intersects") means:
# for each CPA polygon, find every county polygon that it intersects.
#
# This is important because a single CPA polygon can cross county boundaries.
# If we used a stricter assumption too early, we might incorrectly force a CPA
# into only one county before checking how much of it overlaps each county.
#
# The result, `cpa_county_matches`, is therefore an intermediate table:
# - each row is one CPA-to-county match
# - the same CPA_ID may appear multiple times if that CPA intersects multiple counties
# - this table is the starting point for the later "largest overlap area" rule
cpa_county_matches = gpd.sjoin(
    cpa,
    counties,
    how="left",
    predicate="intersects"
)

# Print the shape of the intermediate match table.
# This helps confirm how many CPA-to-county matches were created.
# If the number of rows is much larger than the number of CPAs, that is expected,
# because some CPAs will intersect more than one county.
print("cpa_county_matches shape:", cpa_county_matches.shape)

# Preview the first few rows so we can inspect what the spatial join produced.
# At this stage, we expect to see:
# - CPA_ID from the CPA polygons
# - county fields from the county layer
# - an index_right column indicating which county polygon was matched
#
# This is still an intermediate output, not the final one-county-per-CPA assignment.
display(cpa_county_matches.head())

cpa_county_matches shape: (470356, 7)


,CPA_ID,geometry,index_right,county_fips,county_name,county_name_full,state_fips
0,1,"POLYGON ((-1924591.227 3153922.078, -1924591.227 3153422.078, -1924091.227 3153422.078, -1924091.227 3152422.078, -1925591.227 3152422.078, -1925591.227 315...",1739.0,53073,Whatcom,Whatcom County,53
1,2,"POLYGON ((-1929591.227 3151922.078, -1929591.227 3151422.078, -1929091.227 3151422.078, -1929091.227 3150422.078, -1929591.227 3150422.078, -1929591.227 314...",1739.0,53073,Whatcom,Whatcom County,53
2,3,"POLYGON ((-1900091.227 3148922.078, -1900091.227 3148422.078, -1899591.227 3148422.078, -1899591.227 3146922.078, -1900591.227 3146922.078, -1900591.227 314...",1739.0,53073,Whatcom,Whatcom County,53
3,4,"POLYGON ((-1914091.227 3143922.078, -1914091.227 3142922.078, -1913591.227 3142922.078, -1913591.227 3142422.078, -1915091.227 3142422.078, -1915091.227 314...",1739.0,53073,Whatcom,Whatcom County,53
4,6,"POLYGON ((-1915591.227 3139422.078, -1915591.227 3138922.078, -1916591.227 3138922.078, -1916591.227 3139422.078, -1917091.227 3139422.078, -1917091.227 313...",1739.0,53073,Whatcom,Whatcom County,53


In [27]:
# -----------------------------
# Prepare county geometry for overlap-area calculations
# -----------------------------

# After the spatial join in the previous cell, `cpa_county_matches` contains:
# - the CPA polygon geometry on the left side
# - the county attributes that were matched on the right side
#
# One important detail is that GeoPandas keeps only the *left* geometry by default
# during `sjoin`, which means that right now the dataframe still only has the
# CPA polygon geometry, not the county polygon geometry.
#
# We need the county geometry too because the next step is to calculate:
# - the full area of each CPA polygon
# - the overlap area between each CPA polygon and each county polygon it intersects
# - the share of the CPA that falls inside each county
#
# That is how we decide the "primary county" for each CPA.

# Build a lightweight lookup table containing only county geometry.
# We rename the county geometry column to `county_geometry` so it is very clear
# that this geometry comes from the county layer, not from the CPA layer.
county_geom_lookup = counties[["geometry"]].rename(columns={"geometry": "county_geometry"}).copy()

# Merge the county geometry back into the CPA/county match table.
# The `sjoin` created an `index_right` column, which tells us which county row
# each CPA intersected. We use that to bring in the actual county polygon geometry.
#
# After this merge:
# - `geometry` still refers to the CPA polygon
# - `county_geometry` refers to the matched county polygon
cpa_county_matches = cpa_county_matches.merge(
    county_geom_lookup,
    left_on="index_right",
    right_index=True,
    how="left"
)

# -----------------------------
# Compute overlap metrics
# -----------------------------

# Compute the full area of each CPA polygon in square meters.
# This works correctly because both layers were already reprojected to EPSG:5070,
# which is a projected CRS suitable for area calculations.
cpa_county_matches["cpa_area_m2"] = cpa_county_matches.geometry.area

# Compute the overlap area between the CPA polygon and the matched county polygon.
# This is the critical calculation that tells us how much of each CPA lies inside
# each candidate county.
#
# The result is still in square meters because the geometries are in EPSG:5070.
cpa_county_matches["overlap_area_m2"] = cpa_county_matches.geometry.intersection(
    gpd.GeoSeries(cpa_county_matches["county_geometry"], crs=cpa.crs)
).area

# Convert overlap area to square kilometers for easier interpretation in QA outputs.
cpa_county_matches["overlap_area_km2"] = cpa_county_matches["overlap_area_m2"] / 1_000_000

# Convert full CPA area to square kilometers as well.
cpa_county_matches["cpa_area_km2"] = cpa_county_matches["cpa_area_m2"] / 1_000_000

# Compute the fraction of the CPA that lies inside the candidate county.
# This is useful because some CPAs may intersect multiple counties.
# Later, we will use the largest-overlap rule to choose one primary county per CPA.
cpa_county_matches["overlap_share_of_cpa"] = (
    cpa_county_matches["overlap_area_m2"] / cpa_county_matches["cpa_area_m2"]
)

# -----------------------------
# Preview the overlap results
# -----------------------------

# Show the key fields needed to inspect the overlap logic:
# - CPA_ID: the candidate project area identifier
# - county_fips / county_name / county_name_full / state_fips: the matched county info
# - overlap_area_km2: how much of the CPA overlaps that county
# - overlap_share_of_cpa: what share of the CPA falls in that county
#
# At this stage, the same CPA_ID may appear in multiple rows if that CPA spans
# multiple counties. That is expected. The next step will choose the county with
# the largest overlap as the primary county assignment.
display(
    cpa_county_matches[
        ["CPA_ID", "county_fips", "county_name", "county_name_full", "state_fips", "overlap_area_km2", "overlap_share_of_cpa"]
    ].head(20)
)

,CPA_ID,county_fips,county_name,county_name_full,state_fips,overlap_area_km2,overlap_share_of_cpa
0,1,53073,Whatcom,Whatcom County,53,2.000000,1.000000
1,2,53073,Whatcom,Whatcom County,53,3.750000,1.000000
2,3,53073,Whatcom,Whatcom County,53,2.934317,0.978106
3,4,53073,Whatcom,Whatcom County,53,2.000000,1.000000
4,6,53073,Whatcom,Whatcom County,53,2.250000,1.000000
5,7,53009,Clallam,Clallam County,53,2.750000,1.000000
6,8,53009,Clallam,Clallam County,53,3.250000,1.000000
7,10,53009,Clallam,Clallam County,53,2.250000,1.000000
8,12,53073,Whatcom,Whatcom County,53,2.250000,1.000000
9,13,53073,Whatcom,Whatcom County,53,2.000000,1.000000


In [28]:
# -----------------------------
# Reduce the many possible CPA-to-county matches down to one primary county per CPA
# -----------------------------

# At this point, `cpa_county_matches` may contain multiple rows for the same CPA_ID
# because a single CPA polygon can intersect more than one county polygon.
#
# The goal of this step is to assign exactly one county to each CPA so that the
# final metadata file can be used in downstream clustering workflows that expect
# a single county field per candidate project area.
#
# The rule used here is:
# assign each CPA to the county with the **largest overlap area**.

# First, sort the match table by:
# 1. CPA_ID ascending, so all rows for the same CPA are grouped together
# 2. overlap_area_m2 descending, so the county with the largest overlap for that CPA
#    comes first within each group
#
# After sorting, the top row for each CPA_ID will represent the "best" county match
# according to the largest-overlap rule.
primary_county = (
    cpa_county_matches.sort_values(
        ["CPA_ID", "overlap_area_m2"],
        ascending=[True, False]
    )
    # Keep only the first row for each CPA_ID after sorting.
    # Because of the sort above, this first row is the county with the
    # largest overlap area for that CPA.
    .drop_duplicates(subset=["CPA_ID"], keep="first")
    .copy()
)

# Keep only the final county-assignment fields we want to carry forward.
# These fields include:
# - CPA_ID: the join key back to solar_lcoe_ReEDS.csv
# - county_fips / county_name / county_name_full / state_fips: county identifiers
# - cpa_area_km2: total CPA size
# - overlap_area_km2: size of the overlap with the selected county
# - overlap_share_of_cpa: share of the CPA covered by the selected county
#
# These overlap metrics are useful for QA because they show how strong the
# county assignment was for each CPA.
primary_county = primary_county[
    [
        "CPA_ID",
        "county_fips",
        "county_name",
        "county_name_full",
        "state_fips",
        "cpa_area_km2",
        "overlap_area_km2",
        "overlap_share_of_cpa",
    ]
].reset_index(drop=True)

# Print the final size of the primary county table.
# Ideally, this should now be close to one row per unique CPA_ID.
print("primary_county shape:", primary_county.shape)

# Preview the resulting one-county-per-CPA assignment table.
# This is the main intermediate output that will be merged back into
# solar_lcoe_ReEDS.csv in the next step.
display(primary_county.head(20))

primary_county shape: (406110, 8)


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000


In [29]:
# -----------------------------
# Add centroid longitude/latitude for each CPA polygon
# -----------------------------

# This step is optional for the county-assignment deliverable itself,
# because Jenny’s main request is to add county information into
# `solar_lcoe_ReEDS.csv`.
#
# However, it is still helpful to add centroid longitude/latitude for each CPA because:
# - some downstream clustering workflows may want latitude/longitude as features
# - it gives an easy map point representation for each CPA
# - it provides a simple geographic reference for QA and debugging
#
# Important note:
# These coordinates are NOT being used to assign county here.
# County assignment was already done more accurately using polygon-to-polygon overlap.
# These centroid coordinates are just an additional attribute attached afterward.

# Start from the CPA polygon layer.
# `cpa` currently contains one row per Candidate Project Area polygon,
# with at least:
# - CPA_ID
# - geometry
#
# We copy it so that this centroid calculation does not affect the original `cpa` object.
cpa_centroids = cpa.copy()

# Compute the geometric centroid of each CPA polygon.
# This creates a point at the center of each polygon.
#
# Again, this centroid is NOT used for the county assignment rule.
# It is only being added as a convenient longitude/latitude representation
# of each CPA after the county assignment has already been decided.
cpa_centroids["centroid_geometry"] = cpa_centroids.geometry.centroid

# Build a new GeoDataFrame that keeps:
# - CPA_ID as the unique key
# - centroid_geometry as the active geometry column
#
# The CRS is inherited from `cpa`, which at this point is still in EPSG:5070
# because we reprojected earlier for area-based overlap calculations.
#
# Then we reproject the centroids to EPSG:4326 so the coordinates can be
# stored as normal longitude/latitude values in degrees.
cpa_centroids = gpd.GeoDataFrame(
    cpa_centroids[["CPA_ID"]],
    geometry=cpa_centroids["centroid_geometry"],
    crs=cpa.crs
).to_crs("EPSG:4326")

# Extract longitude from the centroid point geometry.
# In EPSG:4326, x corresponds to longitude.
cpa_centroids["longitude"] = cpa_centroids.geometry.x

# Extract latitude from the centroid point geometry.
# In EPSG:4326, y corresponds to latitude.
cpa_centroids["latitude"] = cpa_centroids.geometry.y

# Keep only the fields we actually want to merge back into the final CPA-to-county table:
# - CPA_ID: join key
# - longitude
# - latitude
#
# This keeps the table small and focused.
cpa_centroids = cpa_centroids[["CPA_ID", "longitude", "latitude"]].copy()

# Merge the centroid longitude/latitude onto the one-county-per-CPA table.
# This keeps the county assignment from the previous step and adds optional
# geographic point coordinates for each CPA.
primary_county = primary_county.merge(
    cpa_centroids,
    on="CPA_ID",
    how="left"
)

# Preview the result so we can confirm that the county-assignment table now
# also includes centroid longitude/latitude.
display(primary_county.head(20))

,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000,-124.683608,48.350214
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000,-124.610383,48.354214
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000,-124.557623,48.299891
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.466001,48.695710
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.257214,48.712506


In [30]:
# -----------------------------
# Add clearer state fields and a plain county column to the CPA-to-county assignment table
# -----------------------------

# At this point, `primary_county` already contains the main county assignment for each CPA,
# including:
# - CPA_ID
# - county_fips
# - county_name
# - county_name_full
# - state_fips
# - overlap metrics
# - centroid longitude/latitude
#
# The goal of this step is to enrich that table with cleaner state fields that are easier
# to use downstream, especially:
# - state_name
# - state_abbrev
#
# This is helpful because the county shapefile gives us state_fips, but later PowerGenome /
# Switch workflows and QA tables are easier to read when we also have plain-text state fields.

# The repo already contains a county GeoJSON used in earlier work:
#   county_layer_for_gui.geojson
#
# That file has convenient fields like:
# - GEOID
# - STATE_NAME
# - STUSPS
#
# We use it here as a lookup table keyed on county_fips so we can attach cleaner state metadata.

if COUNTY_GEOJSON_OPTIONAL.exists():
    # Load the repo's county GeoJSON.
    county_geo = gpd.read_file(COUNTY_GEOJSON_OPTIONAL)

    # Keep only the fields needed for this lookup:
    # - GEOID: county FIPS, which matches the county_fips field in primary_county
    # - STATE_NAME: full state name
    # - STUSPS: two-letter state abbreviation
    #
    # drop_duplicates() is used just in case the GeoJSON contains repeated county entries.
    county_geo = county_geo[["GEOID", "STATE_NAME", "STUSPS"]].drop_duplicates().rename(
        columns={
            "GEOID": "county_fips",
            "STATE_NAME": "state_name",
            "STUSPS": "state_abbrev",
        }
    )

    # Merge the cleaner state fields into the primary county-assignment table using county_fips.
    # After this merge, each CPA should have:
    # - county_fips
    # - county_name
    # - state_name
    # - state_abbrev
    primary_county = primary_county.merge(
        county_geo,
        on="county_fips",
        how="left"
    )

# Add a plain `county` column for downstream clustering configs.
#
# This is important because the PowerGenome renewable clustering documentation/examples
# refer to grouping on a field named `county`. Even though we already have `county_name`,
# adding a plain `county` column makes the output more directly usable in clustering configs
# without requiring a later rename.
primary_county["county"] = primary_county["county_name"]

# Preview the enriched CPA-to-county assignment table.
# At this stage, this table should now contain:
# - the CPA_ID
# - county fields
# - state fields
# - overlap QA metrics
# - centroid longitude/latitude
#
# This is the main intermediate table that will be merged back into solar_lcoe_ReEDS.csv.
display(primary_county.head(20))

,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000,-124.683608,48.350214,Washington,WA,Clallam
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000,-124.610383,48.354214,Washington,WA,Clallam
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000,-124.557623,48.299891,Washington,WA,Clallam
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.466001,48.695710,Washington,WA,Whatcom
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.257214,48.712506,Washington,WA,Whatcom


In [31]:
# -----------------------------
# Merge the one-county-per-CPA assignment back into the solar metadata CSV
# -----------------------------

# `solar_meta` is the original metadata table from:
#   solar_lcoe_ReEDS.csv
#
# It contains the candidate solar site metadata used in the Switch-PG-ReEDS / PowerGenome workflow.
# Earlier in the notebook, we built `primary_county`, which contains one county assignment
# per CPA_ID after doing the polygon-to-polygon spatial join and choosing the county
# with the largest overlap area.
#
# The purpose of this step is to combine those two pieces:
# - keep all original columns from solar_lcoe_ReEDS.csv
# - add the new county and state information that will make county-based clustering possible

# Perform a left merge so that:
# - every original row from solar_meta is preserved
# - the county assignment from primary_county is attached wherever CPA_ID matches
#
# This adds fields such as:
# - county
# - county_name
# - county_name_full
# - county_fips
# - state_fips
# - state_name
# - state_abbrev
# - overlap metrics
# - centroid longitude/latitude
solar_meta_with_county = solar_meta.merge(
    primary_county,
    on="CPA_ID",
    how="left"
)

# Ensure a plain `county` column exists for downstream PowerGenome clustering.
#
# The PowerGenome renewable clustering documentation/examples refer to grouping on `county`,
# so even if the table already has `county_name`, we explicitly make sure there is a field
# named `county` that downstream clustering configs can use directly.
if "county" not in solar_meta_with_county.columns:
    solar_meta_with_county["county"] = solar_meta_with_county["county_name"]

# -----------------------------
# QA / completeness checks
# -----------------------------

print("solar_meta_with_county shape:", solar_meta_with_county.shape)
print("Missing county_fips share:", solar_meta_with_county["county_fips"].isna().mean())
print("Missing county_name share:", solar_meta_with_county["county_name"].isna().mean())
print("Missing county share:", solar_meta_with_county["county"].isna().mean())

display(solar_meta_with_county.head())

solar_meta_with_county shape: (405737, 55)
Missing county_fips share: 2.464650746665944e-06
Missing county_name share: 2.464650746665944e-06
Missing county share: 2.464650746665944e-06


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


In [32]:
# -----------------------------
# Filter the final outputs down to the WECC states only
# -----------------------------

# The full county-enriched output covers the full national CPA dataset,
# which makes the final files quite large.
#
# For this project, we only want to keep the states associated with the
# Western Interconnection / WECC workflow.
#
# Important note:
# this is a state-level filter, not an exact WECC-footprint filter.
# States like Texas and South Dakota are only partially in WECC, but per the
# requested deliverable we are keeping the full listed states.

WECC_STATES = [
    "Arizona",
    "California",
    "Colorado",
    "Idaho",
    "Montana",
    "Nevada",
    "New Mexico",
    "Oregon",
    "South Dakota",
    "Texas",
    "Utah",
    "Washington",
    "Wyoming",
]

# Filter the one-county-per-CPA assignment table down to CPAs whose assigned
# county is in one of the requested WECC states.
#
# This becomes the core filtered reference table for the rest of the outputs.
primary_county_wecc = primary_county[
    primary_county["state_name"].isin(WECC_STATES)
].copy()

# Filter the full solar metadata table down to the same WECC-state subset.
#
# This keeps only the candidate solar metadata rows whose assigned county
# belongs to one of the requested states.
solar_meta_with_county_wecc = solar_meta_with_county[
    solar_meta_with_county["state_name"].isin(WECC_STATES)
].copy()

# Filter the full CPA-to-county overlap table as well so that the QA/audit file
# is also smaller. We do this by keeping only overlap rows whose CPA_ID appears
# in the filtered primary county table.
cpa_county_matches_wecc = cpa_county_matches[
    cpa_county_matches["CPA_ID"].isin(primary_county_wecc["CPA_ID"])
].copy()

# Quick QA checks so it is easy to see how much smaller the WECC-only outputs are.
print("primary_county full shape:", primary_county.shape)
print("primary_county WECC-only shape:", primary_county_wecc.shape)

print("\nsolar_meta_with_county full shape:", solar_meta_with_county.shape)
print("solar_meta_with_county WECC-only shape:", solar_meta_with_county_wecc.shape)

print("\ncpa_county_matches full shape:", cpa_county_matches.shape)
print("cpa_county_matches WECC-only shape:", cpa_county_matches_wecc.shape)

print("\nWECC states present in filtered output:")
print(sorted(solar_meta_with_county_wecc["state_name"].dropna().unique().tolist()))

display(primary_county_wecc.head())
display(solar_meta_with_county_wecc.head())

primary_county full shape: (406110, 13)
primary_county WECC-only shape: (184863, 13)

solar_meta_with_county full shape: (405737, 55)
solar_meta_with_county WECC-only shape: (184776, 55)

cpa_county_matches full shape: (470356, 13)
cpa_county_matches WECC-only shape: (205824, 13)

WECC states present in filtered output:
['Arizona', 'California', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Oregon', 'South Dakota', 'Texas', 'Utah', 'Washington', 'Wyoming']


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


In [33]:
# -----------------------------
# Save the final WECC-only deliverables from the county-assignment workflow
# -----------------------------

# At this point in the notebook, we already created filtered WECC-only tables:
# - primary_county_wecc
# - solar_meta_with_county_wecc
# - cpa_county_matches_wecc
#
# Those are the versions we want to save if the goal is to keep the final outputs
# limited to the requested WECC states only.
#
# If we saved the original tables instead:
# - primary_county
# - solar_meta_with_county
# - cpa_county_matches
# then the output files would still be national and would remain much larger.

# Save the one-county-per-CPA assignment table for the WECC-only subset.
#
# This is a compact QA/reference file showing the final county chosen for each CPA_ID
# after applying the state filter.
primary_county_wecc.to_csv(
    OUTPUT_DIR / "cpa_primary_county_assignment.csv",
    index=False
)

# Save the main WECC-only deliverable:
# the updated solar_lcoe_ReEDS metadata file with county information added.
#
# This file keeps:
# - all original solar metadata columns
# - the added county/state fields
# - overlap metrics
# - centroid longitude/latitude
#
# but only for the requested WECC states.
solar_meta_with_county_wecc.to_csv(
    OUTPUT_DIR / "solar_lcoe_ReEDS_with_county.csv",
    index=False
)

# Save a second copy with a clearer name indicating that the file is intended
# to be used in the downstream clustering workflow.
solar_meta_with_county_wecc.to_csv(
    OUTPUT_DIR / "solar_lcoe_ReEDS_with_county_clustering_ready.csv",
    index=False
)

# Save the full CPA-to-county overlap table for the WECC-only subset.
#
# This is mainly a QA/audit file. It contains all intersecting county matches
# before the one-county-per-CPA rule was applied.
#
# We drop the geometry columns to keep the file smaller and save as parquet
# because parquet is more efficient for larger tables.
cpa_county_matches_wecc.drop(columns=["geometry", "county_geometry"]).to_parquet(
    OUTPUT_DIR / "cpa_county_overlap_full.parquet",
    index=False
)

# -----------------------------
# Build a lightweight run summary for QA and handoff
# -----------------------------

# This summary records:
# - the original national row counts
# - the final WECC-only row counts
# - missingness rates for the new county fields
# - whether useful columns were successfully added
#
# This makes it easy to confirm that the filtering worked and that the final
# metadata file is ready for downstream clustering.
summary = {
    "solar_meta_rows_full": int(len(solar_meta)),
    "solar_meta_rows_wecc_only": int(len(solar_meta_with_county_wecc)),
    "solar_meta_unique_cpa_ids_full": int(solar_meta["CPA_ID"].nunique()),
    "solar_meta_unique_cpa_ids_wecc_only": int(solar_meta_with_county_wecc["CPA_ID"].nunique()),
    "cpa_polygon_rows": int(len(cpa)),
    "county_rows": int(len(counties)),
    "primary_county_rows_wecc_only": int(len(primary_county_wecc)),
    "missing_county_fips_share_wecc_only": float(solar_meta_with_county_wecc["county_fips"].isna().mean()),
    "missing_county_name_share_wecc_only": float(solar_meta_with_county_wecc["county_name"].isna().mean()),
    "missing_county_share_wecc_only": float(solar_meta_with_county_wecc["county"].isna().mean()),
    "has_state_name": "state_name" in solar_meta_with_county_wecc.columns,
    "has_state_abbrev": "state_abbrev" in solar_meta_with_county_wecc.columns,
    "has_longitude": "longitude" in solar_meta_with_county_wecc.columns,
    "has_latitude": "latitude" in solar_meta_with_county_wecc.columns,
    "county_assignment_method": "largest_overlap_area",
    "filtered_states": WECC_STATES,
}

# Save the run summary as JSON so it can be quickly reviewed outside the notebook.
with open(OUTPUT_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Display the summary in the notebook as the final quick QA output.
summary

{'solar_meta_rows_full': 405737,
 'solar_meta_rows_wecc_only': 184776,
 'solar_meta_unique_cpa_ids_full': 405737,
 'solar_meta_unique_cpa_ids_wecc_only': 184776,
 'cpa_polygon_rows': 406110,
 'county_rows': 3235,
 'primary_county_rows_wecc_only': 184863,
 'missing_county_fips_share_wecc_only': 0.0,
 'missing_county_name_share_wecc_only': 0.0,
 'missing_county_share_wecc_only': 0.0,
 'has_state_name': True,
 'has_state_abbrev': True,
 'has_longitude': True,
 'has_latitude': True,
 'county_assignment_method': 'largest_overlap_area',
 'filtered_states': ['Arizona',
  'California',
  'Colorado',
  'Idaho',
  'Montana',
  'Nevada',
  'New Mexico',
  'Oregon',
  'South Dakota',
  'Texas',
  'Utah',
  'Washington',
  'Wyoming']}

## **What this notebook produces**

The main output from this notebook is:

- `data/reeds_county_mapping/outputs/solar_lcoe_ReEDS_with_county.csv`

This is the updated solar metadata file with county information added for each `CPA_ID`, filtered to the requested WECC states. It is the main file to use for the next clustering step.

A second copy is also saved as:

- `data/reeds_county_mapping/outputs/solar_lcoe_ReEDS_with_county_clustering_ready.csv`

This file contains the same data, but with a name that makes its intended use clearer. It includes all original columns from `solar_lcoe_ReEDS.csv`, along with the added county and state fields, overlap metrics, and centroid longitude/latitude.

The notebook also saves two QA files:

- `data/reeds_county_mapping/outputs/cpa_primary_county_assignment.csv`
- `data/reeds_county_mapping/outputs/cpa_county_overlap_full.parquet`

`cpa_primary_county_assignment.csv` contains one final county assignment per `CPA_ID` for the filtered WECC-state subset. `cpa_county_overlap_full.parquet` contains the full CPA-to-county overlap table for the same filtered subset before selecting the primary county.

The main deliverable is the WECC-filtered, county-enriched `solar_lcoe_ReEDS_with_county.csv`.

### **Ignore: Final Data Checks** 

In [34]:
# Confirm one row per CPA_ID in the final WECC-only metadata file
print("total rows:", len(solar_meta_with_county_wecc))
print("unique CPA_IDs:", solar_meta_with_county_wecc["CPA_ID"].nunique())
print("duplicate CPA_ID rows:", solar_meta_with_county_wecc.duplicated(subset=["CPA_ID"]).sum())

total rows: 184776
unique CPA_IDs: 184776
duplicate CPA_ID rows: 0


In [35]:
# Show any rows still missing county fields
missing_rows = solar_meta_with_county_wecc[
    solar_meta_with_county_wecc["county"].isna()
    | solar_meta_with_county_wecc["county_name"].isna()
    | solar_meta_with_county_wecc["county_fips"].isna()
]

print("rows with any missing county info:", len(missing_rows))
display(missing_rows.head(20))

rows with any missing county info: 0


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county


In [36]:
# Inspect potentially ambiguous county assignments
low_overlap = primary_county_wecc[
    primary_county_wecc["overlap_share_of_cpa"] < 0.5
].sort_values("overlap_share_of_cpa")

print("low-overlap CPA assignments:", len(low_overlap))
display(low_overlap.head(20))

low-overlap CPA assignments: 196


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
96676,100642,48465,Val Verde,Val Verde County,48,4.000000,0.000376,0.000094,-101.197199,29.513047,Texas,TX,Val Verde
178350,184003,48323,Maverick,Maverick County,48,4.250000,0.001839,0.000433,-100.521647,28.740355,Texas,TX,Maverick
96737,100703,48479,Webb,Webb County,48,2.250000,0.001169,0.000520,-99.602522,27.633033,Texas,TX,Webb
182837,188490,48505,Zapata,Zapata County,48,5.500000,0.004050,0.000736,-99.339943,26.911892,Texas,TX,Zapata
182838,188491,48505,Zapata,Zapata County,48,4.500000,0.005705,0.001268,-99.218423,26.717322,Texas,TX,Zapata
96742,100708,48505,Zapata,Zapata County,48,3.500000,0.005961,0.001703,-99.248942,26.782915,Texas,TX,Zapata
183959,189612,48479,Webb,Webb County,48,9.199933,0.040332,0.004384,-99.525019,27.342099,Texas,TX,Webb
184041,189694,48479,Webb,Webb County,48,7.300401,0.065814,0.009015,-99.503867,27.409756,Texas,TX,Webb
96753,100719,48061,Cameron,Cameron County,48,3.500000,0.106197,0.030342,-97.555764,25.924918,Texas,TX,Cameron
180531,186184,48323,Maverick,Maverick County,48,5.500000,0.173456,0.031537,-100.363822,28.471060,Texas,TX,Maverick


In [37]:
# Confirm only the requested states remain
print(sorted(solar_meta_with_county_wecc["state_name"].dropna().unique().tolist()))

['Arizona', 'California', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Oregon', 'South Dakota', 'Texas', 'Utah', 'Washington', 'Wyoming']


In [38]:
# Summarize how much of the national file remains after WECC filtering
print("full rows:", len(solar_meta_with_county))
print("WECC-only rows:", len(solar_meta_with_county_wecc))
print("share kept:", len(solar_meta_with_county_wecc) / len(solar_meta_with_county))

full rows: 405737
WECC-only rows: 184776
share kept: 0.4554083063659464


In [39]:
# -----------------------------
# QA check 1: compare CPA_ID coverage across the metadata file and shapefile
# -----------------------------

# This checks whether there are CPA_IDs that appear:
# - in the original solar metadata file but not in the CPA polygon shapefile
# - in the CPA polygon shapefile but not in the metadata file
#
# Small differences are not automatically an error, but this helps explain
# row-count mismatches and gives a clearer picture of coverage.

solar_meta_ids = set(solar_meta["CPA_ID"].dropna().unique())
cpa_ids = set(cpa["CPA_ID"].dropna().unique())

only_in_metadata = solar_meta_ids - cpa_ids
only_in_shapefile = cpa_ids - solar_meta_ids

print("unique CPA_IDs in solar_meta:", len(solar_meta_ids))
print("unique CPA_IDs in CPA shapefile:", len(cpa_ids))
print("CPA_IDs only in solar_meta:", len(only_in_metadata))
print("CPA_IDs only in shapefile:", len(only_in_shapefile))

print("\nExample CPA_IDs only in solar_meta:")
print(sorted(list(only_in_metadata))[:20])

print("\nExample CPA_IDs only in shapefile:")
print(sorted(list(only_in_shapefile))[:20])

unique CPA_IDs in solar_meta: 405737
unique CPA_IDs in CPA shapefile: 406110
CPA_IDs only in solar_meta: 0
CPA_IDs only in shapefile: 373

Example CPA_IDs only in solar_meta:
[]

Example CPA_IDs only in shapefile:
[np.int64(33), np.int64(1838), np.int64(7656), np.int64(7661), np.int64(14379), np.int64(15419), np.int64(15441), np.int64(15442), np.int64(15458), np.int64(15459), np.int64(15470), np.int64(15476), np.int64(15477), np.int64(22358), np.int64(22359), np.int64(22377), np.int64(22383), np.int64(22384), np.int64(22385), np.int64(22411)]


In [40]:
# -----------------------------
# QA check 2: summarize how confident the county assignments look
# -----------------------------

# overlap_share_of_cpa tells us what fraction of the CPA polygon lies in the
# assigned primary county.
#
# Values near 1.0 mean the county assignment is very clear.
# Lower values mean the CPA crosses county boundaries more substantially.

overlap_summary = primary_county_wecc["overlap_share_of_cpa"].describe()
print("Overlap-share summary:")
display(overlap_summary)

# Bucket the overlap shares into easy-to-read ranges.
overlap_buckets = pd.cut(
    primary_county_wecc["overlap_share_of_cpa"],
    bins=[0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0],
    include_lowest=True
).value_counts().sort_index()

print("\nOverlap-share buckets:")
display(overlap_buckets)

Overlap-share summary:


count    184863.000000
mean          0.977511
std           0.083319
min           0.000094
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: overlap_share_of_cpa, dtype: float64


Overlap-share buckets:


overlap_share_of_cpa
(-0.001, 0.25]        21
(0.25, 0.5]          175
(0.5, 0.75]         7680
(0.75, 0.9]         5234
(0.9, 0.99]         5146
(0.99, 1.0]       141022
Name: count, dtype: int64

In [41]:
# -----------------------------
# QA check 3: save potentially ambiguous county assignments
# -----------------------------

# These are the cases where the assigned county covers less than half of the CPA.
# They are not necessarily wrong, but they are the rows most worth reviewing
# if someone wants to inspect edge cases later.

ambiguous_cpas = primary_county_wecc[
    primary_county_wecc["overlap_share_of_cpa"] < 0.5
].sort_values("overlap_share_of_cpa")

print("Potentially ambiguous CPA assignments:", len(ambiguous_cpas))
display(ambiguous_cpas.head(20))

# Save them as a separate QA file for review.
ambiguous_cpas.to_csv(
    OUTPUT_DIR / "cpa_primary_county_assignment_ambiguous_overlap_lt_50pct.csv",
    index=False
)

Potentially ambiguous CPA assignments: 196


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
96676,100642,48465,Val Verde,Val Verde County,48,4.000000,0.000376,0.000094,-101.197199,29.513047,Texas,TX,Val Verde
178350,184003,48323,Maverick,Maverick County,48,4.250000,0.001839,0.000433,-100.521647,28.740355,Texas,TX,Maverick
96737,100703,48479,Webb,Webb County,48,2.250000,0.001169,0.000520,-99.602522,27.633033,Texas,TX,Webb
182837,188490,48505,Zapata,Zapata County,48,5.500000,0.004050,0.000736,-99.339943,26.911892,Texas,TX,Zapata
182838,188491,48505,Zapata,Zapata County,48,4.500000,0.005705,0.001268,-99.218423,26.717322,Texas,TX,Zapata
96742,100708,48505,Zapata,Zapata County,48,3.500000,0.005961,0.001703,-99.248942,26.782915,Texas,TX,Zapata
183959,189612,48479,Webb,Webb County,48,9.199933,0.040332,0.004384,-99.525019,27.342099,Texas,TX,Webb
184041,189694,48479,Webb,Webb County,48,7.300401,0.065814,0.009015,-99.503867,27.409756,Texas,TX,Webb
96753,100719,48061,Cameron,Cameron County,48,3.500000,0.106197,0.030342,-97.555764,25.924918,Texas,TX,Cameron
180531,186184,48323,Maverick,Maverick County,48,5.500000,0.173456,0.031537,-100.363822,28.471060,Texas,TX,Maverick
